# SET50 Dividend Carry

รันทีละเซลล์จากบนลงล่าง (Kernel → Restart & Run All ได้)
ไฟล์ `dividend_api.py`, `dividend_model.py`, `set50.py`, `dividend_run.py`
ต้องอยู่โฟลเดอร์เดียวกับ notebook นี้

In [ ]:
# 1. Setup: ตรวจไฟล์ แล้ว import
import importlib
import sys
from pathlib import Path

import pandas as pd

MODEL_DIR = Path.cwd()   # โฟลเดอร์ที่มีไฟล์ .py ทั้ง 4 ไฟล์
REQUIRED = ["dividend_api.py", "dividend_model.py", "set50.py", "dividend_run.py"]
missing = [name for name in REQUIRED if not (MODEL_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f"ไม่พบ {missing} ใน {MODEL_DIR} — copy ไฟล์มาไว้ที่นี่ หรือแก้ MODEL_DIR"
    )
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import dividend_api, dividend_model, set50, dividend_run
for module in (dividend_api, dividend_model, set50, dividend_run):
    importlib.reload(module)

from dividend_api import fetch_dividends
from dividend_model import ModelParams
from set50 import (
    membership_table, change_membership, membership_changes,
    default_instruments,
)
from dividend_run import run_snapshot, backtest_snapshot, backtest_many

print("model", dividend_model.MODEL_VERSION, "| pandas", pd.__version__)

In [ ]:
# 2. Membership scenario
SET50_NOW = [
    "ADVANC", "AOT", "AWC", "BANPU", "BBL", "BCP", "BDMS", "BEM", "BH", "BJC",
    "CCET", "COM7", "CPALL", "CPF", "CPN", "CRC", "DELTA", "EGCO", "GPSC", "GULF",
    "HMPRO", "IVL", "KBANK", "KKP", "KTB", "KTC", "LH", "MINT", "MRDIYT", "MTC",
    "OR", "OSP", "PTT", "PTTEP", "PTTGC", "RATCH", "SCB", "SCC", "SCGP", "TCAP",
    "TFG", "THAI", "TIDLOR", "TISCO", "TLI", "TOP", "TRUE", "TTB", "TU", "WHA",
]

# ค่าเริ่มต้น: ครึ่งปีถัดไปสมาชิกเหมือนปัจจุบัน
MEMBERSHIP = membership_table(SET50_NOW, "2026H2", through="2027H1")

# bet การเปลี่ยนสมาชิก (ตัวอย่าง — แก้ตามที่คิด แล้วตั้งชื่อ SCENARIO ให้ตรง)
# MEMBERSHIP = change_membership(MEMBERSHIP, "2027H1", add=["XYZ"], remove=["THAI"])
# หรือแก้ทีละช่อง: MEMBERSHIP.loc["THAI", "2027H1"] = False
SCENARIO = "base"

display(membership_changes(MEMBERSHIP))

In [ ]:
# 3. ดึงข้อมูลจาก API (ทุก symbol ใน scenario)
TOKEN = "วาง token ของวันนี้ที่นี่"   # ใส่ทั้งก้อน จะมีหรือไม่มีคำว่า "Bearer " ก็ได้

if TOKEN.removeprefix("Bearer ").count(".") != 4:
    print("คำเตือน: token ควรมี 5 ส่วนคั่นด้วยจุด — อาจ copy มาไม่ครบ")

dividend_raw, fetch_log = fetch_dividends(MEMBERSHIP.index.tolist(), token=TOKEN)
print(f"{len(dividend_raw):,} rows")
display(fetch_log.loc[fetch_log["status"].ne("OK")])

In [ ]:
# 4. Snapshot วันนี้ -> Excel
AS_OF = pd.Timestamp.today().normalize()
SET_HOLIDAYS = ["2026-12-31", "2027-01-01"]   # ใส่วันหยุด SET ให้ครบ
INSTRUMENTS = ["S50Z26", "S50H27", "S50M27", "S50Z26H27", "S50H27M27"]
# หรือ INSTRUMENTS = default_instruments(AS_OF, SET_HOLIDAYS)

snapshot = run_snapshot(
    dividend_raw,
    as_of=AS_OF,
    membership=MEMBERSHIP,
    instruments=INSTRUMENTS,
    holidays=SET_HOLIDAYS,
    scenario=SCENARIO,
    output_dir="dividend_model_output",
)
print(snapshot["path"])
display(snapshot["instruments"])

In [ ]:
# 5. ตรวจผล
display(snapshot["matrix"])                                   # sheet Carry
display(snapshot["events"].groupby("status").size().rename("rows").to_frame())
display(snapshot["review"])

## Backtest (ไม่บังคับ)
ใส่รายชื่อ SET50 ที่ใช้จริงของรอบนั้นแทน `SET50_NOW`

In [ ]:
BET = membership_table(SET50_NOW, "2025H2", through="2026H1")
result = backtest_snapshot(
    dividend_raw,
    as_of="2025-10-01",
    membership=BET,
    instruments=["S50Z25", "S50H26", "S50Z25H26"],
    holidays=SET_HOLIDAYS,
)
display(result["carry"].groupby("instrument")[["forecast", "actual"]].sum())
display(result["events"]["outcome"].value_counts())